In [ ]:
import requests
from bs4 import BeautifulSoup
import csv
import calendar
import time

session = requests.Session()

# Step 1: Get CSRF token from form page
form_url = "https://cestat.gov.in/order-status"
resp = session.get(form_url, verify=True)
soup = BeautifulSoup(resp.text, "html.parser")
csrf_token = soup.find("input", {"name": "csrf_token"})["value"]

post_url = "https://cestat.gov.in/ajax/order-status-web"

results = []

['124438', '109120', '129525', '104044', '133568', '107079', '136507', '119315', '127482']

# Step 2: Loop through months of 2025
for month in range(1, 13):
    from_date = f"01-{month:02d}-2025"
    last_day = calendar.monthrange(2025, month)[1]
    to_date = f"{last_day:02d}-{month:02d}-2025"

    payload = {
        "tab": 4,
        "csrf_token": csrf_token,
        "bench": 127482,
        "from": from_date,
        "to": to_date,
        "captcha_code": "111111",   # safe to leave blank
        "data_table_length": 200
    }

    response = session.post(post_url, data=payload, verify=True)
    response_json = response.json()

    # Step 3: Parse JSON rows
    for row in response_json.get("data", []):
        serial = row[0]
        case_no = row[1]
        parties = row[2].replace("<br>", " vs ")
        date = row[3]

        # Extract href from HTML
        soup = BeautifulSoup(row[4], "html.parser")
        link_tag = soup.find("a")
        pdf_url = None
        if link_tag and link_tag.get("href"):
            href = link_tag["href"].replace("./", "")
            pdf_url = f"https://cestat.gov.in/{href}"

        results.append({
            "month": month,
            "serial": serial,
            "case_no": case_no,
            "parties": parties,
            "date": date,
            "pdf_url": pdf_url
        })
    
    time.sleep(5)   

# Step 4: Save results to CSV
with open(f"cestat_cases_{payload["bench"]}_2025.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["month", "serial", "case_no", "parties", "date", "pdf_url"])
    writer.writeheader()
    writer.writerows(results)

# print("Saved results to cestat_cases_4_2025.csv")

In [2]:
%load_ext autoreload
%autoreload 2
import pprint, json, math, os, sys, camelot
# sys.path.append(os.path.abspath(dir_path))


import fitz, pdfplumber, ocrmypdf, pprint
import pandas as pd
import numpy as np
from collections import defaultdict
from app.parse_table import TableParser
from app.utils import Helper
# from app.parse_regex import *
# from app.parse_table import *

dry_path = r'DryRun.pdf'
fin_path = r'\data\input\financial_indices.xlsx'
# mutual_fund = Helper.get_fund_paths(fund_path)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
helper = Helper()
path = r"C:\Users\kaustubh.keny\Downloads\Factsheet Oct 2025\Aditya Birla Sun Life Mutual Fund\3_31-Oct-25_FS.pdf"
helper.get_all_pdf_data(path)

In [ ]:
lines = [
    ((110, 0), (110, 812)),# Vertical line
    ((0, 350), (812, 350)),
    ((570, 0), (570, 812))
]
pages = [12, 14,16]
bboxes = [[0, 120, 180, 812],[180, 85, 360, 812]] #[(0, 85, 180, 812),(180, 85, 360, 812),(0,100,270,812),(0,100,350,812)]
pages = [i for i in range(1,110)]
Helper.draw_lines_on_pdf(sample_path, lines, bboxes, pages, dry_path)

In [6]:
#MIRAE
mir_path = r"C:\Users\kaustubh.keny\Downloads\Factsheet Oct 2025\Mirae Asset Mutual Fund\27_31-Oct-25_1_FS.pdf"
table_parser = TableParser()
tables = camelot.read_pdf(mir_path,flavor="stream",pages="74-90")
dfs = pd.concat([table.df for table in tables], ignore_index=True)

sr1 = table_parser.get_matching_row_indices(dfs,["MIRAE Asset","ETF"],thresh=3)
sr2 = table_parser.get_matching_row_indices(dfs,["investment objective"],thresh=1)

print(sr1,sr2)

[0, 2, 4, 72, 74, 76, 149, 220, 289, 292, 359, 363, 431, 434, 500, 505, 568, 576, 623, 629, 634, 688, 742, 748, 749, 798, 803, 808, 809, 813, 814, 858, 923, 981] [8, 9, 14, 15, 81, 82, 88, 153, 154, 159, 224, 225, 231, 232, 296, 297, 302, 304, 367, 368, 373, 374, 375, 438, 439, 445, 513, 514, 518, 523, 524, 577, 578, 582, 585, 630, 631, 635, 684, 685, 689, 692, 750, 751, 755, 810, 811, 816, 869, 870, 874, 877, 929, 930, 935, 990, 991, 996, 998, 1041, 1042, 1046, 1050]


In [8]:
dfs.to_csv("tp.csv")

In [ ]:
#NIPPON
# nip_jan = r"C:\Users\Kaustubh.keny\Projects\PDF\Jan 25\Nippon India Mutual Fund\33_31-Jan-25_FS.pdf"
# nip_feb = ""
# nip_mar = r"C:\Users\Kaustubh.keny\Projects\PDF\Mar 25\Nippon India Mutual Fund\33_31-Mar-25_FS.pdf"
# nip_apr = r"C:\Users\Kaustubh.keny\Projects\PDF\Apr 25\Nippon India Mutual Fund\33_30-Apr-25_FS.pdf"
# table_parser = TableParser()
# tables = camelot.read_pdf(nip_apr,flavor="stream",pages="129-140")
# dfs = pd.concat([table.df for table in tables], ignore_index=True)

# sr1 = table_parser._get_matching_row_indices(dfs,["Nippon.+?Fund","Scheme\\s*Name"],thresh=2)
# sr2 = table_parser._get_matching_row_indices(dfs,["minimum application"],thresh=1)
# sr2_expanded = {i for idx in sr2 for i in range(idx, idx + 5)}
# sr1_expanded = {i for idx in sr1 for i in range(idx, idx + 1)}

# all_indices = sr1_expanded | sr2_expanded
# valid_indices = sorted(i for i in all_indices if i in dfs.index)

# filtered_df = dfs.loc[valid_indices].reset_index()
# for idx, rows in filtered_df.iterrows():
#     row_val = " ".join([str(i) for i in rows])
#     row_val = SidKimRegex()._normalize_alphanumeric(row_val)
#     # print(row_val)
#     matches = re.findall(r"Nippon.+?(?:Funds?|ETF|Path|Saver|active|financial|allocation|tunities|duration|psu debt|advantage|small cap 250)\s*(?:of Funds?|Fund of Funds?|Funds?|.+?Plan|FoF)?",row_val, re.IGNORECASE)
#     if matches:
#         print(idx, len(matches))
#         print(matches)

In [ ]:
#BAJAJAJ
# jan = r"C:\Users\Kaustubh.keny\Projects\PDF\Jan 25\Bajaj finserv Mutual Fund\59_31-Jan-25_FS.pdf"
# feb = r"C:\Users\Kaustubh.keny\Projects\PDF\Feb 25\Bajaj finserv Mutual Fund\59_28-Feb-25_FS.pdf"
# mar = r"C:\Users\Kaustubh.keny\Projects\PDF\Mar 25\Bajaj finserv Mutual Fund\59_31-Mar-25_FS.pdf"
# apr = r"C:\Users\Kaustubh.keny\Projects\PDF\Apr 25\Bajaj finserv Mutual Fund\59_30-Apr-25_FS.pdf"


# table_parser = TableParser()
# tables = camelot.read_pdf(apr,flavor="hybrid",pages="11-14")
# dfs = pd.concat([table.df for table in tables], ignore_index=True)
# sc1 = table_parser.get_matching_col_indices(dfs,["Bajaj.+?Fund","SCHEME\\s*NAME"],thresh=20)
# sc2 = table_parser.get_matching_col_indices(dfs,["Jensen","Standard\\s*Deviation","Information\\s*ratio","Portfolio\\s*Quants","Tracking Error","YTM","Average\\s*Maturity","Sharpe"],thresh=10)
# sc3 = table_parser.get_matching_col_indices(dfs,["Average\\s*Maturity","Modified Duration","Macaulay Duration","YTM"], thresh=8)
# all_cols = sorted(set(sc1)) + [sc2[0],sc2[0]+1, sc3[0],sc3[0]+1]
# fdf = dfs.iloc[:, all_cols]
# fdf.columns = ["MUTUAL_FUND"] + [f"METRICS_{i}" for i in range(1, fdf.shape[1])]
# bajaj = re.compile(
#     r"(Bajaj?\\s*finserv.+?)",
#     re.IGNORECASE
# )
# fdf.MUTUAL_FUND = table_parser.clean_series(fdf.MUTUAL_FUND,["normalize_alphanumeric"])
# fdf.MUTUAL_FUND = fdf.MUTUAL_FUND.apply(lambda x: bajaj.findall(x)[0] if isinstance(x, str) and bajaj.findall(x) else "")
# fdf = table_parser.clean_dataframe(fdf,["newline_to_space","str_to_pd_NA"])
# fdf = fdf.dropna(axis=0, how="all").dropna(axis=1, how="all")
# fdf =table_parser.clean_dataframe(fdf,['NA_to_str'])

# data = {}
# for idx, rows in fdf.iterrows():
#     values = list(rows)
#     main_scheme_name = str(values[0]).strip() if not pd.isna(values[0]) else ""
#     if main_scheme_name:
#         temp = main_scheme_name
#         if temp not in data:
#             data[temp] = {"metrics": []}
#         data[temp]["metrics"].append(" ".join(map(str, values[1:])))
    
#     if temp:
#         data[temp]["metrics"].append(" ".join(map(str, values)))


In [ ]:
# #HDFC
# table_parser = TableParser()
# tables = camelot.read_pdf(hdfc1,flavor="lattice",pages="91-93")
# dfs = pd.concat([table.df for table in tables], ignore_index=True)
# sc1 = table_parser._get_matching_col_indices(dfs,["HDFC.+?Fund"],thresh=20)
# sc2 = table_parser._get_matching_col_indices(dfs,["MINIMUM\\s*APPLICATION\\s*AMOUNT","Additional\\s*Purchase"], thresh=20)

# print("Matched columns:", sc1,sc2)
# all_cols = list(set(sc1 + sc2))
# fdf = dfs.iloc[:, all_cols]
# fdf.columns = ["MUTUAL_FUND","MIN_ADD"]
# hdfc_pattern = re.compile(
#     r"(HDFC.+?(?:FUNDS?|ETF|PATH|INDEX|SAVER)\s*(?:OF FUNDS?|FUND OF FUNDS|FOF|.+?PLAN)?)",
#     re.IGNORECASE
# )
# fdf.MUTUAL_FUND = table_parser._clean_series(fdf.MUTUAL_FUND,["normalize_alphanumeric"])
# fdf.MUTUAL_FUND = fdf.MUTUAL_FUND.apply(lambda x: hdfc_pattern.findall(x)[0] if isinstance(x, str) and hdfc_pattern.findall(x) else x)
# fdf = table_parser._clean_dataframe(fdf,["newline_to_space","str_to_pd_NA"])
# fdf = fdf.dropna(axis=0, how="all").dropna(axis=1, how="all")
# fdf =table_parser._clean_dataframe(fdf,['NA_to_str'])

# data = {}
# for idx, rows in fdf.iterrows():
#     values = list(rows)
#     main_scheme_name = values[0]
#     if main_scheme_name not in data:
#         data[main_scheme_name] = {"min_add":values[1]}
#     else:
#         data[main_scheme_name].update({"min_add_one":values[1]})

In [ ]:
# #DSP
# jan = r"C:\Users\Kaustubh.keny\Projects\PDF\Jan 25\DSP Mutual Fund\8_31-Jan-25_FS.pdf"
# feb = r"C:\Users\Kaustubh.keny\Projects\PDF\Feb 25\DSP Mutual Fund\8_28-Feb-25_FS.pdf"
# mar = r"C:\Users\Kaustubh.keny\Projects\PDF\Mar 25\DSP Mutual Fund\8_31-Mar-25_FS.pdf"
# apr = r"C:\Users\Kaustubh.keny\Projects\PDF\Apr 25\DSP Mutual Fund\8_30-Apr-25_FS.pdf"

# table_parser = TableParser()
# tables = camelot.read_pdf(apr,flavor="lattice",pages="107-123")
# dfs = pd.concat([table.df for table in tables], ignore_index=True)
# sc1 = table_parser._get_matching_col_indices(dfs,["DSP.+?Fund"],thresh=12)
# sc2 = table_parser._get_matching_col_indices(dfs,["REGULAR\\s+PLAN","DIRECT\\s+PLAN"], thresh=12)
# sc3 = table_parser._get_matching_col_indices(dfs,["Managing this scheme","total work experience"],thresh=12)
# print("Matched columns:", sc1,sc2,sc3)
# all_cols = list(set(sc1 + sc2 + sc3))
# fdf = dfs.iloc[:, all_cols]
# fdf["LOAD_STRUCTURE"] = fdf.iloc[:, -1]
# fdf.columns = ["MUTUAL_FUND","FUND_MANAGER","MIN_ADD","LOAD_STRUCTURE"]

# dsp_pattern = re.compile(
#     r"(DSP.+?(?:FUNDS?|ETF|PATH|INDEX|SAVER)\s*(?:OF FUNDS?|FUNDs?|FUND OF FUNDS?|FOF|.+?PLAN)?)",
#     re.IGNORECASE
# )

# fdf.MUTUAL_FUND = table_parser._clean_series(fdf.MUTUAL_FUND,["normalize_alphanumeric"])
# fdf.MUTUAL_FUND = fdf.MUTUAL_FUND.apply(lambda x: dsp_pattern.findall(x)[0] if isinstance(x, str) and dsp_pattern.findall(x) else pd.NA)
# fdf = table_parser._clean_dataframe(fdf,["newline_to_space","str_to_pd_NA"])
# fdf = fdf.dropna(axis=0, how="all").dropna(axis=1, how="all")
# fdf =table_parser._clean_dataframe(fdf,['NA_to_str'])

# data = {}
# for idx, rows in fdf.iterrows():
#     values = list(rows)
#     main_scheme_name = values[0]
#     if main_scheme_name not in data:
#         data[main_scheme_name] = {"fund_manager":values[1],"load_structure":values[3],"min_add":values[2]}
#     else:
#          data[main_scheme_name].update({"fund_manager_one":values[1],"load_structure_one":values[3],"min_add_one":values[2]})
        


In [ ]:
import fitz
import pytesseract
from PIL import Image
import io, re

def get_proper_fund_names(path: str):
    title = {}
    pattern ="((?:LI?i?C|BSE|BANK|SMALL|HEALTH|MNEY|[aA]n\\s*open).*?(?:FUND|Path|ETF|FTF|EOF|FOF|PLAN|SAVER|tax saving scheme|small cap stocks)\\s*(?:FUND\\s*OF\\s*FUND)?)"
    with fitz.open(path) as doc:
        for pgn, page in enumerate(doc):
            clip = fitz.Rect(300, 0, 595, 80)
            pix = page.get_pixmap(clip=clip, dpi=300)
            img = Image.open(io.BytesIO(pix.tobytes()))
            text = pytesseract.image_to_string(img)
            cleaned = re.sub("[^A-Za-z0-9\\s\\.,\\-\\(\\)\\+\\%\\:\\&]+", "", text).strip()
            if matches := re.findall(pattern, cleaned, re.DOTALL):
                title[pgn] = " ".join([_ for _ in matches[0].strip().split(" ") if _])
                print(f"{pgn}:matched {matches[0]}")
            # print(f"[OCR] Page {pgn}: {cleaned}")
            # if cleaned:
            #     title[pgn] = cleaned
    return title
path = r"C:\Users\kaustubh.keny\OneDrive - Cogencis Information Services Ltd\Documents\MUTUAL FUND FACTSHEET FY19-25\2021_changed\LIC Mutual Fund\25_31-Dec-21_FS.pdf"
title = get_proper_fund_names(path)

In [ ]:
"""
21_31-Dec-25_FS.pdf	20/01/26, 13:10	completed	kaustubh.keny	-	✔️	JSON	docs	
autorenew
2	21_31-Dec-25_FS.pdf	20/01/26, 13:10	completed	kaustubh.keny	-	✔️	JSON	docs	
autorenew
3	9_31-Dec-25_FS.pdf	20/01/26, 13:04	completed	kaustubh.keny	-	-	JSON	docs	
autorenew
4	9_31-Dec-25_FS.pdf	20/01/26, 13:04	completed	kaustubh.keny	-	-	JSON	docs	
autorenew
5	9_31-Dec-25_FS.pdf	20/01/26, 13:04	completed	kaustubh.keny	-	-	JSON	docs	
autorenew
6	13_31-Dec-25_FS.pdf	20/01/26, 09:58	completed	manoj.kadam	-	✔️	JSON	docs	
autorenew
7	27_31-Dec-25_1_FS.pdf	20/01/26, 09:56	completed	tanvi.salunkhe	-	✔️	JSON	docs	
autorenew
8	27_31-Dec-25_FS.pdf	20/01/26, 09:55	completed	tanvi.salunkhe	-	✔️	JSON	docs	
autorenew
9	5_31-Dec-25_FS.pdf	20/01/26, 09:31	completed	manoj.kadam	-	✔️	JSON	docs	
autorenew
10	101_31-Dec-25_FS.pdf	19/01/26, 13:42	completed	kaustubh.keny	-	-	JSON	docs	
autorenew
11	16_31-Dec-25_1_FS.pdf	19/01/26, 12:42	failed	tanvi.salunkhe	ValueError: Unknown AMC ID or File.	-	-	-	
autorenew
12	16_31-Dec-25_FS.pdf	19/01/26, 12:37	completed	tanvi.salunkhe	-	✔️	JSON	docs	
autorenew
13	3_31-Dec-25_FS.pdf	19/01/26, 12:09	completed	manoj.kadam	-	✔️	JSON	docs	
autorenew
14	26_31-Dec-25_FS.pdf	16/01/26, 16:50	completed	tanvi.salunkhe	-	✔️	JSON	docs	
autorenew
15	40_31-Dec-25_FS.pdf	14/01/26, 16:53	completed	kaustubh.keny	-	-	JSON	docs	
autorenew

"""

In [2]:
import re, fitz

def get_proper_fund_names(path: str, pattern:str):
    title = {} 
    with fitz.open(path) as doc:
        for pgn, page in enumerate(doc):
            text = " ".join(page.get_text("text", clip = (160, 640, 600, 812)).split("\n")) #clip = (0, 0, 210, 155)
            text = re.sub("[^A-Za-z0-9\\s\\.,\\-\\(\\)\\+\\%\\:\\&]+", "", text).strip()
            print(f"{pgn}:-{text}")
            if matches := re.findall(pattern, text, re.DOTALL):
                title[pgn] = " ".join([_ for _ in matches[0].strip().split(" ") if _ ])
                print(pgn,matches[0])
    return title
path = r"C:\Users\kaustubh.keny\Projects\OFFICE PROJECTS\mywork-repo\35_31-Dec-25_FS.pdf"
pattern = "((?:SBI|i\\s*_|S35).*?(?:Fund\\s*(?:\\-?\\s*Investment\\s*Plan|\\-?\\s*Savings\\s*Plan)?|Index|Saver|ETF|FTF|F[oO]F)\\s*(?:of [Ff]unds?|.*?Aggressive\\s*Plan|.*?Hybrid\\s*Plan|.*?Conservative\\s*Plan)?)"
# pattern = "(MIRAE.*?)NSE\\s*[Ss]ymbol"
title = get_proper_fund_names(path,pattern)

0:-
1:-.......................................................................................................................... .......................................................................................................................... .......................................................................................................................... .......................................................................................................................... .......................................................................................................................... lution Oriented Scheme  d Equity - Thematic Equity - Thematic m Balanced Fund) Hybrid - Aggressive Hybrid Fund brid Fund) Hybrid - Conservative Hybrid Fund m Monthly Income Plan - Floater) Hybrid - Multi Asset Allocation Fund 27 28 29 31 32 33
2:-ll Schemes P) Funds ix Branches ...................................................................................................

In [ ]:
def _normalize_alphanumeric(text: str) -> str:
    if not isinstance(text,str):
        return text
    text = re.sub(r"[^a-zA-Z0-9]+", " ", str(text))
    return re.sub(r"\s+", " ", text).strip().lower()

def extract_clipped_data(input:str, pages:list, bboxes:list):
        
        document = fitz.open(input)
        final_list = []
    
        for pgn in pages:
            page = document[pgn]
            
            all_blocks = [] #store every data from bboxes
            
            for bbox in bboxes:
                blocks, seen_blocks = [], set()  #store unique blocks based on content and bbox
                
                page_blocks = page.get_text('dict', clip=bbox)['blocks']
                for block in page_blocks:
                    if block['type'] == 0 and 'lines' in block: #type 0 means text block
                        #hash_key
                        block_key = (tuple(block['bbox']), tuple(tuple(line['spans'][0]['text'] for line in block['lines'])))
                        if block_key not in seen_blocks:
                            seen_blocks.add(block_key)
                            blocks.append(block)

                sorted_blocks = sorted(blocks, key=lambda x: (x['bbox'][1], x['bbox'][0]))
                all_blocks.append(sorted_blocks)

            final_list.append({
                "pgn": pgn,
                "block": all_blocks #will be list[list,list,..]
            })

        document.close()
        return final_list
    
def extract_data_relative_line(path: str, line_x: float, side: str):
    doc = fitz.open(path)
    pages = doc.page_count

    final_list = []

    for pgn in range(pages):
        page = doc[pgn]

        blocks = page.get_text("dict")["blocks"]
        sorted_blocks = sorted(blocks, key=lambda x: (x["bbox"][1], x["bbox"][0]))
        extracted_blocks = []

        # Keep track of blocks to avoid duplicates
        added_blocks = set()

        for block in sorted_blocks:
            block_id = id(block)  # Unique identifier for the block

            for line in block.get("lines", []):
                for span in line.get("spans", []):
                    origin = span["origin"]
                    x0, _ = origin

                    # Check the side condition
                    if side == "left" and x0 < line_x and block_id not in added_blocks:
                        extracted_blocks.append(block)
                        added_blocks.add(block_id)  # Mark block as added
                    elif side == "right" and x0 > line_x and block_id not in added_blocks:
                        extracted_blocks.append(block)
                        added_blocks.add(block_id)  # Mark block as added

      
        final_list.append({
            "pgn": pgn,
            "blocks": extracted_blocks
        })

    doc.close()

    return final_list
  
def get_clipped_data(input:str, bboxes:list[set], *args):
    
        document = fitz.open(input)
        final_list = []
        if args:
            pages = list(args)
        else:
            pages = [i for i in document.page_count]
        
        for pgn in pages:
            page = document[pgn]

            blocks = []
            for bbox in bboxes:
                blocks.extend(page.get_text('dict', clip = bbox)['blocks']) #get all blocks
            
            filtered_blocks = [block for block in blocks if block['type']== 0 and 'lines' in block]
            # sorted_blocks = sorted(filtered_blocks, key= lambda x: (x['bbox'][1], x['bbox'][0]))
             # Extract text from sorted blocks
            extracted_text = []
            for block in filtered_blocks:
                block_text = []
                for line in block['lines']:
                    line_text = " ".join(span['text'] for span in line['spans'])
                    block_text.append(line_text)
                extracted_text.append("\n".join(block_text))
            
            final_list.append({
            "pgn": pgn,
            "block": filtered_blocks,
            "text": extracted_text
            })
            
            
        document.close()
        return final_list
    
def extract_clipped_text_all_pages(pdf_path, clip_coords):
    results = {}
    doc = fitz.open(pdf_path)
    clip_rect = fitz.Rect(*clip_coords)
    try:
        for page_number, page in enumerate(doc):
            text = page.get_text("text", clip=clip_rect).strip()
            results[page_number] = text
    finally:
        doc.close()
    return results

In [ ]:
from tabula.io import read_pdf
import pandas as pd


def pdf_to_excel(pdf_file_path, excel_file_path):
    # Read PDF file
    tables = read_pdf(pdf_file_path, pages='9-16') #json format here
    
    #pymupdf
    #get the table title based on page of the table, then pair order wise
    #save that in a dict and use that in the below writer ??
    
    # Write each table to a separate sheet in the Excel file
    with pd.ExcelWriter(excel_file_path) as writer:
        for i, table in enumerate(tables):
            table.to_excel(writer, sheet_name=f'Sheet{i+1}')


pdf_path =r"C:\Users\kaustubh.keny\Downloads\IFSCA\ifsca-bulletin-april-june-202306102023042858.pdf"
pdf_to_excel(pdf_path, 'ifsca-bulletin-april-june.xlsx')

In [10]:
from tabula.io import read_pdf

pdf_path = r"C:\Users\kaustubh.keny\Downloads\IFSCA\ifsca-bulletin-april-june-202306102023042858.pdf"

# Extract tables as JSON
tables_json = read_pdf(pdf_path, pages="9-16", output_format="json", multiple_tables=True)

# Print the raw JSON
import json
with open("hello.json","w+") as f:
    json.dump(tables_json,f)

In [11]:
from tabula.io import read_pdf
import fitz, pandas as pd, re

pdf_path = r"C:\Users\kaustubh.keny\Downloads\IFSCA\ifsca-bulletin-april-june-202306102023042858.pdf"

def extract_titles(pdf_path, page_range):
    doc = fitz.open(pdf_path)
    titles = {}
    for page_num in page_range:
        page = doc[page_num-1]
        blocks = page.get_text("blocks")
        titles[page_num] = [b[4] for b in blocks if re.match(r"Table\s+\d+:.*", b[4])]
    return titles

def extract_tables(pdf_path, page_range):
    tables = {}
    for page in page_range:
        tables_df = read_pdf(pdf_path, pages=page, multiple_tables=True)
        tables[page] = tables_df
    return tables

def pair_titles_tables(tables, titles):
    paired = []
    for page in tables:
        page_tables = tables[page]
        page_titles = titles.get(page, [])
        for i, table in enumerate(page_tables):
            title = page_titles[i] if i < len(page_titles) else None
            paired.append({"page": page, "df": table, "title": title})
    return paired

def pdf_to_excel(pdf_file_path, excel_file_path, page_range):
    titles = extract_titles(pdf_file_path, page_range)
    tables = extract_tables(pdf_file_path, page_range)
    paired = pair_titles_tables(tables, titles)

    with pd.ExcelWriter(excel_file_path) as writer:
        for i, p in enumerate(paired):
            sheet_name = f"Sheet{i+1}"
            # Write table starting at row 2
            p["df"].to_excel(writer, sheet_name=sheet_name, startrow=1, index=False)
            ws = writer.sheets[sheet_name]
            if p["title"]:
                # Write title in first row, merged across all columns
                ws.merge_range(0, 0, 0, len(p["df"].columns)-1, p["title"])

# Usage
page_range = range(9, 17)
pdf_to_excel(pdf_path, "ifsca-april-june.xlsx", page_range)